# Module 2.3: Baum-Welch Algorithm - EM Training for HMMs

## Learning Objectives
By completing this notebook, you will:
1. Understand the Expectation-Maximization (EM) framework for latent variable models
2. Implement the complete Baum-Welch algorithm for HMM parameter learning
3. Learn both discrete and Gaussian emission distributions from data
4. Apply model selection techniques (BIC/AIC) to choose the number of states
5. Train HMMs on financial time series to discover hidden market regimes

## Prerequisites
- Forward-backward algorithm (Module 2.1)
- EM algorithm concepts
- Maximum likelihood estimation

## Estimated Time: 70 minutes

---

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
learning_objectives([
    "Forward-backward algorithm (Module 2.1)",
    "EM algorithm concepts",
    "Maximum likelihood estimation"
])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Tuple, List
from scipy import stats

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

## 1. The Learning Problem

**Given:** Observation sequence O = (o₁, o₂, ..., oₜ)

**Find:** Parameters λ* = (π*, A*, B*) that maximize P(O|λ)

$$\lambda^* = \arg\max_{\lambda} P(O | \lambda)$$

### The Challenge

- States are **hidden** - we can't directly count transitions
- We can't compute MLE like supervised learning
- Need iterative approach: **Expectation-Maximization**

### EM Solution

1. **E-step:** Compute expected sufficient statistics given current parameters
2. **M-step:** Update parameters to maximize expected log-likelihood
3. **Iterate** until convergence

## 2. The Baum-Welch Algorithm

Baum-Welch is EM specialized for HMMs.

### E-Step: Compute Expected Statistics

Using forward-backward, compute:

**State occupation probability:**
$$\gamma_t(i) = P(q_t = i | O, \lambda)$$

**Transition probability:**
$$\xi_t(i,j) = P(q_t = i, q_{t+1} = j | O, \lambda)$$

### M-Step: Update Parameters

**Initial state distribution:**
$$\hat{\pi}_i = \gamma_1(i)$$

**Transition matrix:**
$$\hat{a}_{ij} = \frac{\sum_{t=1}^{T-1} \xi_t(i,j)}{\sum_{t=1}^{T-1} \gamma_t(i)}$$

**Emission (Discrete):**
$$\hat{b}_i(v_k) = \frac{\sum_{t: o_t=v_k} \gamma_t(i)}{\sum_{t=1}^T \gamma_t(i)}$$

**Emission (Gaussian):**
$$\hat{\mu}_i = \frac{\sum_{t=1}^T \gamma_t(i) \cdot o_t}{\sum_{t=1}^T \gamma_t(i)}$$

$$\hat{\sigma}_i^2 = \frac{\sum_{t=1}^T \gamma_t(i) \cdot (o_t - \hat{\mu}_i)^2}{\sum_{t=1}^T \gamma_t(i)}$$

## 3. Complete Baum-Welch Implementation

We'll build a complete HMM class with training.

In [ ]:
# Purpose: Implement complete Baum-Welch HMM class
# Key Concept: EM algorithm for parameter learning

class GaussianHMM:
    """
    Hidden Markov Model with Gaussian emissions trained via Baum-Welch.
    """
    
    def __init__(self, n_states: int, random_seed: int = None):
        """
        Initialize HMM with random parameters.
        
        Parameters
        ----------
        n_states : int
            Number of hidden states
        """
        if random_seed is not None:
            np.random.seed(random_seed)
        
        self.K = n_states
        
        # Step 1: Initialize parameters randomly
        self.pi = np.ones(self.K) / self.K  # Uniform initial distribution
        
        # Random transition matrix (row-stochastic)
        self.A = np.random.rand(self.K, self.K)
        self.A = self.A / self.A.sum(axis=1, keepdims=True)
        
        # Gaussian emission parameters
        self.means = np.random.randn(self.K) * 0.01
        self.stds = np.ones(self.K) * 0.02
    
    def _emission_prob(self, observation: float, state: int) -> float:
        """
        Compute P(observation | state) for Gaussian emission.
        """
        return stats.norm.pdf(
            observation,
            loc=self.means[state],
            scale=self.stds[state]
        )
    
    def forward(self, observations: np.ndarray) -> Tuple[np.ndarray, np.ndarray, float]:
        """
        Scaled forward algorithm.
        
        Returns
        -------
        alpha : ndarray (T, K)
            Scaled forward variables
        scaling : ndarray (T,)
            Scaling factors
        log_likelihood : float
            Log P(O|λ)
        """
        T = len(observations)
        alpha = np.zeros((T, self.K))
        scaling = np.zeros(T)
        
        # Initialization
        for i in range(self.K):
            alpha[0, i] = self.pi[i] * self._emission_prob(observations[0], i)
        scaling[0] = np.sum(alpha[0])
        alpha[0] /= scaling[0]
        
        # Induction
        for t in range(1, T):
            for j in range(self.K):
                alpha[t, j] = np.sum(alpha[t-1] * self.A[:, j]) * \
                              self._emission_prob(observations[t], j)
            scaling[t] = np.sum(alpha[t])
            alpha[t] /= scaling[t]
        
        log_likelihood = np.sum(np.log(scaling))
        return alpha, scaling, log_likelihood
    
    def backward(self, observations: np.ndarray, scaling: np.ndarray) -> np.ndarray:
        """
        Scaled backward algorithm.
        """
        T = len(observations)
        beta = np.zeros((T, self.K))
        
        # Initialization
        beta[-1] = 1.0 / scaling[-1]
        
        # Induction (backward)
        for t in range(T-2, -1, -1):
            for i in range(self.K):
                beta[t, i] = np.sum(
                    self.A[i, :] *
                    np.array([self._emission_prob(observations[t+1], j) for j in range(self.K)]) *
                    beta[t+1, :]
                )
            beta[t] /= scaling[t]
        
        return beta
    
    def compute_gamma_xi(
        self,
        observations: np.ndarray,
        alpha: np.ndarray,
        beta: np.ndarray
    ) -> Tuple[np.ndarray, np.ndarray]:
        """
        Compute gamma and xi (E-step).
        """
        T = len(observations)
        
        # Gamma: P(q_t = i | O)
        gamma = alpha * beta
        gamma /= gamma.sum(axis=1, keepdims=True)
        
        # Xi: P(q_t = i, q_{t+1} = j | O)
        xi = np.zeros((T-1, self.K, self.K))
        
        for t in range(T-1):
            denominator = 0.0
            for i in range(self.K):
                for j in range(self.K):
                    xi[t, i, j] = (
                        alpha[t, i] *
                        self.A[i, j] *
                        self._emission_prob(observations[t+1], j) *
                        beta[t+1, j]
                    )
                    denominator += xi[t, i, j]
            xi[t] /= (denominator + 1e-10)
        
        return gamma, xi
    
    def m_step(self, observations: np.ndarray, gamma: np.ndarray, xi: np.ndarray):
        """
        M-step: Update parameters.
        """
        T = len(observations)
        
        # Step 1: Update initial distribution
        self.pi = gamma[0]
        
        # Step 2: Update transition matrix
        for i in range(self.K):
            for j in range(self.K):
                numerator = np.sum(xi[:, i, j])
                denominator = np.sum(gamma[:-1, i]) + 1e-10
                self.A[i, j] = numerator / denominator
        
        # Normalize rows
        self.A = self.A / (self.A.sum(axis=1, keepdims=True) + 1e-10)
        
        # Step 3: Update Gaussian emission parameters
        for i in range(self.K):
            # Weighted mean
            gamma_sum = np.sum(gamma[:, i]) + 1e-10
            self.means[i] = np.sum(gamma[:, i] * observations) / gamma_sum
            
            # Weighted variance
            diff = observations - self.means[i]
            self.stds[i] = np.sqrt(
                np.sum(gamma[:, i] * diff**2) / gamma_sum
            )
    
    def fit(
        self,
        observations: np.ndarray,
        max_iterations: int = 100,
        tolerance: float = 1e-4,
        verbose: bool = True
    ) -> List[float]:
        """
        Fit HMM parameters using Baum-Welch (EM) algorithm.
        
        Parameters
        ----------
        observations : ndarray (T,)
            Observation sequence
        max_iterations : int
            Maximum EM iterations
        tolerance : float
            Convergence threshold for log-likelihood improvement
        verbose : bool
            Print progress
        
        Returns
        -------
        log_likelihoods : list of float
            Log-likelihood at each iteration
        """
        log_likelihoods = []
        
        for iteration in range(max_iterations):
            # E-step: Forward-Backward
            alpha, scaling, log_likelihood = self.forward(observations)
            beta = self.backward(observations, scaling)
            gamma, xi = self.compute_gamma_xi(observations, alpha, beta)
            
            # M-step: Update parameters
            self.m_step(observations, gamma, xi)
            
            log_likelihoods.append(log_likelihood)
            
            if verbose and (iteration % 10 == 0 or iteration == max_iterations - 1):
                print(f"Iteration {iteration:3d}: log P(O|λ) = {log_likelihood:.4f}")
            
            # Check convergence
            if iteration > 0:
                improvement = log_likelihoods[-1] - log_likelihoods[-2]
                if improvement < tolerance:
                    if verbose:
                        print(f"\nConverged after {iteration+1} iterations")
                    break
        
        return log_likelihoods
    
    def predict(self, observations: np.ndarray) -> np.ndarray:
        """
        Predict most likely states using Viterbi algorithm.
        """
        T = len(observations)
        
        # Log-space Viterbi
        log_delta = np.zeros((T, self.K))
        psi = np.zeros((T, self.K), dtype=int)
        
        # Initialization
        for i in range(self.K):
            log_delta[0, i] = np.log(self.pi[i] + 1e-10) + \
                             np.log(self._emission_prob(observations[0], i) + 1e-10)
        
        # Induction
        for t in range(1, T):
            for j in range(self.K):
                probs = log_delta[t-1] + np.log(self.A[:, j] + 1e-10)
                psi[t, j] = np.argmax(probs)
                log_delta[t, j] = probs[psi[t, j]] + \
                                 np.log(self._emission_prob(observations[t], j) + 1e-10)
        
        # Backtrack
        states = np.zeros(T, dtype=int)
        states[-1] = np.argmax(log_delta[-1])
        
        for t in range(T-2, -1, -1):
            states[t] = psi[t+1, states[t+1]]
        
        return states

print("GaussianHMM class implemented successfully!")

## 4. Testing Baum-Welch on Synthetic Data

Generate data from a known HMM and see if Baum-Welch recovers the parameters.

In [ ]:
# Purpose: Test parameter recovery on synthetic data
# Key Concept: Verify EM algorithm correctness

# Generate synthetic data from known HMM
np.random.seed(42)

# True parameters
true_hmm = GaussianHMM(n_states=2, random_seed=123)
true_hmm.means = np.array([0.001, -0.002])  # Bull: +0.1%, Bear: -0.2%
true_hmm.stds = np.array([0.01, 0.02])      # Bull: 1% vol, Bear: 2% vol
true_hmm.A = np.array([[0.95, 0.05],
                       [0.10, 0.90]])
true_hmm.pi = np.array([0.7, 0.3])

# Simulate observations
def simulate_hmm(hmm, T):
    states = [np.random.choice(hmm.K, p=hmm.pi)]
    observations = [np.random.normal(hmm.means[states[0]], hmm.stds[states[0]])]
    
    for _ in range(T-1):
        states.append(np.random.choice(hmm.K, p=hmm.A[states[-1]]))
        observations.append(
            np.random.normal(hmm.means[states[-1]], hmm.stds[states[-1]])
        )
    
    return np.array(states), np.array(observations)

true_states, observations = simulate_hmm(true_hmm, 500)

print("TRUE PARAMETERS:")
print("="*60)
print(f"Means: {true_hmm.means}")
print(f"Stds:  {true_hmm.stds}")
print(f"\nTransition matrix:\n{true_hmm.A}")
print(f"\nInitial distribution: {true_hmm.pi}")

# Learn parameters using Baum-Welch
print("\n" + "="*60)
print("LEARNING PARAMETERS WITH BAUM-WELCH:")
print("="*60 + "\n")

learned_hmm = GaussianHMM(n_states=2, random_seed=456)
log_likelihoods = learned_hmm.fit(observations, max_iterations=100, verbose=True)

print("\n" + "="*60)
print("LEARNED PARAMETERS:")
print("="*60)
print(f"Means: {learned_hmm.means}")
print(f"Stds:  {learned_hmm.stds}")
print(f"\nTransition matrix:\n{learned_hmm.A}")
print(f"\nInitial distribution: {learned_hmm.pi}")

# Predict states
predicted_states = learned_hmm.predict(observations)

# Note: States might be permuted (0↔1)
accuracy = max(
    np.mean(predicted_states == true_states),
    np.mean(predicted_states == 1 - true_states)
)

print(f"\nState prediction accuracy: {accuracy:.1%}")

## 5. Visualizing Convergence

In [ ]:
# Purpose: Visualize EM convergence and results
# Key Concept: Monitor log-likelihood improvement

fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Plot 1: Log-likelihood convergence
ax1 = axes[0]
ax1.plot(log_likelihoods, 'o-', linewidth=2, markersize=6)
ax1.set_xlabel('Iteration', fontsize=12)
ax1.set_ylabel('Log P(O|λ)', fontsize=12)
ax1.set_title('Baum-Welch Convergence', fontsize=14)
ax1.grid(True, alpha=0.3)

# Plot 2: True vs predicted states
ax2 = axes[1]
time = np.arange(len(true_states))
ax2.fill_between(time, true_states, alpha=0.5, label='True States', step='mid')
# Predicted states might be permuted
if np.mean(predicted_states == true_states) < 0.5:
    predicted_states = 1 - predicted_states
ax2.fill_between(time, predicted_states, alpha=0.5, label='Predicted States', step='mid')
ax2.set_xlabel('Time', fontsize=12)
ax2.set_ylabel('State', fontsize=12)
ax2.set_title('State Sequence: True vs Predicted', fontsize=14)
ax2.set_yticks([0, 1])
ax2.set_yticklabels(['Bull', 'Bear'])
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

# Plot 3: Observations colored by predicted state
ax3 = axes[2]
for state in range(2):
    mask = predicted_states == state
    label = 'Bull' if state == 0 else 'Bear'
    ax3.scatter(time[mask], observations[mask], label=f'Predicted {label}',
               alpha=0.6, s=20)
ax3.set_xlabel('Time', fontsize=12)
ax3.set_ylabel('Return', fontsize=12)
ax3.set_title('Observations Colored by Predicted Regime', fontsize=14)
ax3.legend(fontsize=11)
ax3.axhline(0, color='black', linestyle='--', alpha=0.3)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Exercise 1: Implement K-Means Initialization

**Task:** Instead of random initialization, use K-means clustering to initialize HMM parameters.

This often leads to faster convergence and better solutions.

**Expected Output:** Improved initialization function that uses clustering.

In [ ]:
# YOUR CODE HERE
# ---------------

from sklearn.cluster import KMeans

def initialize_with_kmeans(observations: np.ndarray, n_states: int) -> GaussianHMM:
    """
    Initialize HMM parameters using K-means clustering.
    
    Parameters
    ----------
    observations : ndarray (T,)
        Observation sequence
    n_states : int
        Number of states
    
    Returns
    -------
    hmm : GaussianHMM
        HMM with K-means initialized parameters
    """
    # YOUR IMPLEMENTATION HERE
    # Hint: Use KMeans to cluster observations, then:
    # - Initialize means/stds from cluster statistics
    # - Initialize transition matrix from cluster sequence
    # - Initialize pi from initial cluster assignments
    
    return None

# Test initialization
hmm_kmeans = initialize_with_kmeans(observations, 2)

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_1():
    assert hmm_kmeans is not None, "Don't forget to implement initialize_with_kmeans!"
    assert isinstance(hmm_kmeans, GaussianHMM), "Should return a GaussianHMM object"
    
    # Check parameters are reasonable
    assert np.all(hmm_kmeans.stds > 0), "Standard deviations should be positive"
    assert np.allclose(hmm_kmeans.A.sum(axis=1), 1.0), "Transition rows should sum to 1"
    assert np.isclose(hmm_kmeans.pi.sum(), 1.0), "Initial distribution should sum to 1"
    
    # Check means are different from random initialization
    random_hmm = GaussianHMM(n_states=2, random_seed=999)
    assert not np.allclose(hmm_kmeans.means, random_hmm.means), \
        "K-means initialization should differ from random"
    
    print("✅ Exercise 1 passed!")
    print(f"\nK-means initialized means: {hmm_kmeans.means}")
    print(f"K-means initialized stds:  {hmm_kmeans.stds}")

test_exercise_1()

## Exercise 2: Model Selection with BIC

**Task:** Implement BIC (Bayesian Information Criterion) to select the optimal number of states.

$$\text{BIC} = -2 \log L + k \log n$$

where k = number of parameters, n = number of observations.

Lower BIC is better.

**Expected Output:** Function that finds the best number of states.

In [ ]:
# YOUR CODE HERE
# ---------------

def compute_bic(log_likelihood: float, n_params: int, n_observations: int) -> float:
    """
    Compute Bayesian Information Criterion.
    """
    # YOUR IMPLEMENTATION HERE
    return None

def select_num_states(
    observations: np.ndarray,
    max_states: int = 5,
    n_init: int = 3
) -> Tuple[int, List[float]]:
    """
    Select number of states using BIC.
    
    Parameters
    ----------
    observations : ndarray (T,)
        Observation sequence
    max_states : int
        Maximum number of states to try
    n_init : int
        Number of random initializations per K
    
    Returns
    -------
    best_K : int
        Optimal number of states
    bics : list of float
        BIC for each K
    """
    # YOUR IMPLEMENTATION HERE
    # For each K from 2 to max_states:
    #   - Fit HMM with multiple random initializations
    #   - Keep best log-likelihood
    #   - Compute BIC
    # Return K with lowest BIC
    
    return None, None

# Test model selection
best_K, bics = select_num_states(observations, max_states=4)

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_2():
    assert best_K is not None, "Don't forget to implement select_num_states!"
    assert bics is not None, "BICs should not be None"
    
    # Check best_K is in valid range
    assert 2 <= best_K <= 4, "Best K should be between 2 and 4"
    
    # Check BICs list
    assert len(bics) == 3, "Should have BICs for K=2,3,4"
    assert all(isinstance(b, (int, float)) for b in bics), "BICs should be numbers"
    
    # Best K should correspond to minimum BIC
    min_bic_idx = np.argmin(bics)
    assert best_K == min_bic_idx + 2, "Best K should have lowest BIC"
    
    print("✅ Exercise 2 passed!")
    print(f"\nBIC values:")
    for k, bic in enumerate(bics, start=2):
        marker = " <- BEST" if k == best_K else ""
        print(f"  K={k}: BIC={bic:.2f}{marker}")
    print(f"\nOptimal number of states: {best_K}")

test_exercise_2()

## Exercise 3: Real Market Data Application

**Task:** Download actual market data and fit an HMM to discover hidden regimes.

Use S&P 500 returns or simulate realistic market data.

**Expected Output:** Trained HMM with identified Bull/Bear/Neutral regimes.

In [ ]:
# Generate realistic market data (or use real data)
np.random.seed(789)

# Simulate 2 years of daily data
T_market = 500
market_hmm = GaussianHMM(n_states=2, random_seed=100)
market_hmm.means = np.array([0.0005, -0.0010])  # Bull: 0.05%, Bear: -0.10%
market_hmm.stds = np.array([0.008, 0.020])      # Bull: 0.8% vol, Bear: 2.0% vol
market_hmm.A = np.array([[0.97, 0.03],
                         [0.05, 0.95]])          # Persistent regimes
market_hmm.pi = np.array([0.6, 0.4])

market_states, market_returns = simulate_hmm(market_hmm, T_market)

# YOUR CODE HERE
# ---------------

# Step 1: Fit HMM to market returns
fitted_market_hmm = None  # Replace with your implementation

# Step 2: Decode regimes
decoded_regimes = None  # Replace with your implementation

# Step 3: Analyze regime characteristics
# - Compute average return in each regime
# - Compute volatility in each regime
# - Identify regime transitions

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_3():
    assert fitted_market_hmm is not None, "Don't forget to fit the HMM!"
    assert decoded_regimes is not None, "Don't forget to decode regimes!"
    
    # Check HMM was trained
    assert hasattr(fitted_market_hmm, 'means'), "HMM should have means"
    assert len(decoded_regimes) == T_market, "Should decode all time steps"
    
    # Check regime characteristics make sense
    regime_0_returns = market_returns[decoded_regimes == 0]
    regime_1_returns = market_returns[decoded_regimes == 1]
    
    mean_0 = np.mean(regime_0_returns)
    mean_1 = np.mean(regime_1_returns)
    
    # One regime should have positive mean, one negative
    assert (mean_0 > 0 and mean_1 < 0) or (mean_0 < 0 and mean_1 > 0), \
        "Regimes should have different return characteristics"
    
    print("✅ Exercise 3 passed!")
    print(f"\nRegime characteristics:")
    print(f"  Regime 0: mean={mean_0:.4f}, std={np.std(regime_0_returns):.4f}")
    print(f"  Regime 1: mean={mean_1:.4f}, std={np.std(regime_1_returns):.4f}")
    
    # Visualize
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    
    ax1 = axes[0]
    ax1.plot(np.cumsum(market_returns), 'k-', alpha=0.7)
    ax1.set_ylabel('Cumulative Return')
    ax1.set_title('Market Performance')
    ax1.grid(True, alpha=0.3)
    
    ax2 = axes[1]
    ax2.fill_between(range(T_market), decoded_regimes, alpha=0.6, step='mid')
    ax2.set_xlabel('Time (days)')
    ax2.set_ylabel('Regime')
    ax2.set_title('Decoded Market Regimes')
    ax2.set_yticks([0, 1])
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

test_exercise_3()

## Summary

### Key Takeaways

1. **Baum-Welch is EM for HMMs** - alternates between computing expected statistics (E) and updating parameters (M)
2. **Guaranteed to improve likelihood** each iteration (or stay constant)
3. **Converges to local optimum** - use multiple random initializations
4. **K-means initialization** often leads to better solutions than random
5. **BIC for model selection** - choose number of states that balances fit and complexity

### Parameter Update Intuition

- **π**: Expected initial state probabilities
- **A**: Expected transitions divided by expected state occupancy
- **Means**: Weighted average of observations (weights = state probabilities)
- **Variances**: Weighted variance around learned means

### Common Pitfalls

1. **Label switching** - States 0 and 1 are arbitrary; might flip between runs
2. **Local optima** - Always use multiple initializations
3. **Overfitting** - Too many states fit noise; use BIC/AIC
4. **Numerical underflow** - Always use scaling in forward-backward

### What's Next

In Module 3, we'll extend to multivariate Gaussian HMMs for modeling multiple correlated time series.

---

**Next Module:** Module 3 - Gaussian HMMs with multivariate emissions